# 综合工程实践

学习目标：能交付一个可安装的数据汇总模块，并验证纯计算、异步文件处理与消费者导入。

前置知识：数组与 Map、输入校验、JSON、ES 模块、Promise、测试和 npm 包入口。

适用版本：ECMAScript 2025、Node.js 24.11.0；.mjs 使用 ES 模块。

环境准备：[环境配置与运行](README.md)。

工作目录：content/编程语言/javascript。

配套脚本：位于 scripts/26-data-pipeline/。

1. [package.json](scripts/26-data-pipeline/package.json)：分发内容与 ESM 公共入口。
2. [validate.mjs](scripts/26-data-pipeline/validate.mjs)：统一校验与无副作用规范化。
3. [index.mjs](scripts/26-data-pipeline/index.mjs)：可复用的纯汇总函数。
4. [io.mjs](scripts/26-data-pipeline/io.mjs)：并发文件处理与等待写入。
5. [data-a.json](scripts/26-data-pipeline/data-a.json)：第一份实际输入。
6. [data-b.json](scripts/26-data-pipeline/data-b.json)：第二份实际输入。
7. [demo.mjs](scripts/26-data-pipeline/demo.mjs)：实际读写、输出比对和临时文件清理。
8. [pipeline.test.mjs](scripts/26-data-pipeline/pipeline.test.mjs)：计算边界与真实文件集成测试。
9. [consumer.mjs](scripts/26-data-pipeline/consumer.mjs)：安装消费者中的包名导入与来源断言。
10. [pack-check.mjs](scripts/26-data-pipeline/pack-check.mjs)：真实 npm pack、安装、消费者运行和清理。

Step 1：运行模块自动化测试。

```bash
npm run test:26
```

Step 2：运行实际文件汇总。

```bash
npm run demo:26
```

Step 3：打包并在临时消费者中安装验证。

```bash
npm run pack:26
```

## 1 接口契约与包结构

本例汇总记录中的金额。每条记录有 group（非空分类字符串）、amountCents（以分为单位的非负安全整数）、active（是否参与统计的布尔值）。先校验全部记录，再筛选 active 为 true 的记录，按分类统计条数和金额；结果按总额降序，同额按分类字符串升序。

接口 summarizeRows 只做计算；createReport 负责并发读取 JSON 文件与写入 JSON 报告。使用整数分避免本例中小数金额相加的误差，但累计值仍须检查安全整数范围。空输入得到空数组，非法记录不能静默当成零。

这个可安装子包不依赖开发工具；files 只收录三个实现文件，exports 公开纯函数入口与 ./io 子入口。测试、示例和输入样本不进入分发包。private 阻止发布到 registry，本章只在本地 pack 和安装，不擅自声明许可证。

配套 [package.json](scripts/26-data-pipeline/package.json)：

```json
{
  "name": "notebook-data-summary",
  "version": "1.0.0",
  "private": true,
  "type": "module",
  "engines": { "node": "24.11.0" },
  "files": ["index.mjs", "validate.mjs", "io.mjs"],
  "exports": { ".": "./index.mjs", "./io": "./io.mjs" }
}
```

## 2 校验并规范化输入

validateRows 返回新记录数组，分类名去除首尾空白，并只保留接口字段。校验函数不主动修改调用者的数组与记录。每个字段只读取一次，校验和后续计算使用同一份值，避免 getter 在重复读取时返回不同结果。不能只用 JSON.parse 的成功判断数据有效，也不能用真值转换把字符串“false”当成布尔假。

错误消息包含记录索引，索引从 0 开始。使用数组 entries 的 for...of 而非只依赖 every 或 map，能让稀疏数组中的缺项以 undefined 进入校验，避免跳过空洞。

配套 [validate.mjs](scripts/26-data-pipeline/validate.mjs)：

```javascript
export function validateRows(rows) {
  if (!Array.isArray(rows)) throw new TypeError("rows must be an array");
  const normalized = [];
  for (const [index, row] of rows.entries()) {
    if (row === null || typeof row !== "object" || Array.isArray(row)) {
      throw new TypeError(`row ${index}: expected object`);
    }
    const { group, amountCents, active } = row;
    if (typeof group !== "string" || group.trim() === "") {
      throw new TypeError(`row ${index}: group must be nonempty`);
    }
    if (!Number.isSafeInteger(amountCents) || amountCents < 0) {
      throw new TypeError(`row ${index}: amountCents must be nonnegative safe integer`);
    }
    if (typeof active !== "boolean") {
      throw new TypeError(`row ${index}: active must be boolean`);
    }
    normalized.push({ group: group.trim(), amountCents, active });
  }
  return normalized;
}
```

## 3 筛选、分组、统计与稳定输出

Map 用分类字符串作为键，避免把分类名称当作对象原型上的属性。先过滤不活跃记录，再累计 count 与 totalCents。输出数组是新建的，可独立排序；同额时显式比较 group，使结果不依赖文件完成顺序或区域排序配置。

这里的字符串比较按语言的字符串关系比较规则，不承诺符合任何自然语言词典顺序；若产品要求中文排序，应另指定 Intl.Collator 的区域设置与兼容条件。金额累加超过安全整数范围时抛 RangeError，阻止输出精度不可靠的报告。

配套 [index.mjs](scripts/26-data-pipeline/index.mjs)：

```javascript
import { validateRows } from "./validate.mjs";
export { validateRows };

export function summarizeRows(rows) {
  const groups = new Map();
  for (const row of validateRows(rows)) {
    if (!row.active) continue;
    const summary = groups.get(row.group) ?? { group: row.group, count: 0, totalCents: 0 };
    const nextTotal = summary.totalCents + row.amountCents;
    if (!Number.isSafeInteger(nextTotal)) throw new RangeError("group total exceeds safe integer");
    summary.count += 1;
    summary.totalCents = nextTotal;
    groups.set(row.group, summary);
  }
  return [...groups.values()].sort((left, right) => {
    if (left.totalCents !== right.totalCents) return right.totalCents - left.totalCents;
    return left.group < right.group ? -1 : left.group > right.group ? 1 : 0;
  });
}
```

## 4 异步读取、失败汇总与 JSON 输出

createReport 接收输入路径数组 inputPaths 与输出路径 outputPath。readFile 和 writeFile 属于 Node.js 文件 API；编码显式为 UTF-8。输入文件很小且数量由本例固定，直接并发读取；大批量输入应使用有限并发，不能机械扩展为同时打开任意数量文件。

每个文件的读取、JSON 解析、根数组与记录字段校验都放在一个异步任务中。allSettled 等待这些任务结束，把失败汇总为 AggregateError；每个文件保留首个错误，外层消息注明路径，cause 保留原始原因。

只有全部输入通过后才合并计算；跨文件金额累加溢出时另抛 RangeError，不属于读取任务的错误汇总。写入 Promise 必须等待，写入失败则直接向上传播。输出路径由调用者指定，应选择独立目标，避免覆盖输入；本例所有运行入口使用独立临时目录。

配套 [io.mjs](scripts/26-data-pipeline/io.mjs)：

```javascript
import { readFile, writeFile } from "node:fs/promises";
import { summarizeRows, validateRows } from "./index.mjs";

export async function createReport(inputPaths, outputPath) {
  const results = await Promise.allSettled(inputPaths.map(async (path) => {
    try {
      const rows = JSON.parse(await readFile(path, "utf8"));
      if (!Array.isArray(rows)) throw new TypeError("file must contain an array");
      return validateRows(rows);
    } catch (cause) {
      throw new Error(`input ${path} failed`, { cause });
    }
  }));
  const errors = results.filter((item) => item.status === "rejected").map((item) => item.reason);
  if (errors.length > 0) throw new AggregateError(errors, "input files failed");
  const rows = results.flatMap((item) => item.value);
  const report = summarizeRows(rows);
  await writeFile(outputPath, JSON.stringify(report, null, 2) + "\n", "utf8");
  return report;
}
```

配套 [data-a.json](scripts/26-data-pipeline/data-a.json)：

```json
[
  { "group": "books", "amountCents": 1200, "active": true },
  { "group": "tools", "amountCents": 900, "active": false }
]
```

配套 [data-b.json](scripts/26-data-pipeline/data-b.json)：

```json
[
  { "group": " books ", "amountCents": 800, "active": true },
  { "group": "tools", "amountCents": 2500, "active": true }
]
```

配套 [demo.mjs](scripts/26-data-pipeline/demo.mjs)：

```javascript
import assert from "node:assert/strict";
import { mkdtemp, readFile, rm } from "node:fs/promises";
import { resolve, relative, isAbsolute } from "node:path";
import { createReport } from "./io.mjs";

const temporaryRoot = import.meta.dirname;
const directory = await mkdtemp(resolve(temporaryRoot, "js-c-data-"));
try {
  const outputPath = resolve(directory, "summary.json");
  const report = await createReport([
    new URL("./data-a.json", import.meta.url),
    new URL("./data-b.json", import.meta.url),
  ], outputPath);
  const expected = [
    { group: "tools", count: 1, totalCents: 2500 },
    { group: "books", count: 2, totalCents: 2000 },
  ];
  assert.deepEqual(report, expected);
  assert.deepEqual(JSON.parse(await readFile(outputPath, "utf8")), expected);
  console.log(JSON.stringify(report)); // → tools 共 2500 分，books 共 2000 分且有两条记录
} finally {
  const child = relative(temporaryRoot, directory);
  assert.ok(child.startsWith("js-c-data-") && !isAbsolute(child) && !child.includes(".."));
  await rm(directory, { recursive: true });
}
console.log("report written read back and cleaned"); // → report written read back and cleaned
```

## 5 对公共行为进行自动化测试

纯函数测试覆盖筛选、规范化、排序、不修改输入、空数组、非法数据、金额溢出，以及重复访问时改变返回值的 getter。I/O 测试使用真实临时文件，断言输出内容，并在读取失败或多个文件字段无效时验证没有生成输出文件。

测试 finally 负责清理，既覆盖成功路径也覆盖失败路径。AggregateError 的 errors 数组保存各文件的错误包装，cause 保留底层错误；测试分别核对文件路径、ENOENT、SyntaxError 和字段错误。样本只涉及本地文件，无远程请求或随意等待。

配套 [pipeline.test.mjs](scripts/26-data-pipeline/pipeline.test.mjs)：

```javascript
import test from "node:test";
import assert from "node:assert/strict";
import { mkdtemp, writeFile, readFile, access, rm } from "node:fs/promises";
import { resolve, relative, isAbsolute } from "node:path";
import { summarizeRows, validateRows } from "./index.mjs";
import { createReport } from "./io.mjs";

test("summary: groups filter sort and purity", () => {
  const rows = [
    { group: " b ", amountCents: 100, active: true },
    { group: "a", amountCents: 100, active: true },
    { group: "a", amountCents: 50, active: false },
  ];
  const before = JSON.stringify(rows);
  assert.deepEqual(summarizeRows(rows), [
    { group: "a", count: 1, totalCents: 100 },
    { group: "b", count: 1, totalCents: 100 },
  ]);
  assert.equal(JSON.stringify(rows), before);
  assert.notEqual(validateRows(rows)[0], rows[0]);
});
test("summary: empty zero and invalid boundaries", () => {
  assert.deepEqual(summarizeRows([]), []);
  assert.equal(summarizeRows([{ group: "zero", amountCents: 0, active: true }])[0].totalCents, 0);
  assert.throws(() => summarizeRows(null), /rows must be an array/);
  assert.throws(() => summarizeRows(new Array(1)), /row 0: expected object/);
  const valid = { group: "a", amountCents: 10, active: true };
  for (const patch of [{ group: " " }, { amountCents: -1 }, { amountCents: NaN }, { active: "false" }]) {
    assert.throws(() => summarizeRows([{ ...valid, ...patch }]), TypeError);
  }
  assert.throws(() => summarizeRows([
    { ...valid, amountCents: Number.MAX_SAFE_INTEGER }, valid,
  ]), /group total exceeds safe integer/);
});
for (const [field, laterValue] of [["group", ""], ["amountCents", -1], ["active", false]]) {
  test(`summary: read ${field} once`, () => {
    const row = { group: " a ", amountCents: 10, active: true };
    const firstValue = row[field];
    let reads = 0;
    Object.defineProperty(row, field, {
      get() { return ++reads <= (field === "amountCents" ? 2 : 1) ? firstValue : laterValue; },
    });
    assert.deepEqual(summarizeRows([row]), [{ group: "a", count: 1, totalCents: 10 }]);
    assert.equal(reads, 1);
  });
}

test("report: real write and grouped failures", async () => {
  const root = import.meta.dirname;
  const directory = await mkdtemp(resolve(root, "js-c-test-"));
  try {
    const input = resolve(directory, "input.json");
    const output = resolve(directory, "output.json");
    await writeFile(input, '[{"group":"a","amountCents":20,"active":true}]', "utf8");
    assert.deepEqual(await createReport([input], output), [{ group: "a", count: 1, totalCents: 20 }]);
    assert.equal(JSON.parse(await readFile(output, "utf8"))[0].totalCents, 20);
    const absentOutput = resolve(directory, "absent.json");
    await assert.rejects(createReport([resolve(directory, "missing-a.json"), resolve(directory, "missing-b.json")], absentOutput), (error) => {
      assert.ok(error instanceof AggregateError);
      assert.equal(error.errors.length, 2);
      assert.ok(error.errors.every((item) => item.cause.code === "ENOENT"));
      return true;
    });
    await assert.rejects(access(absentOutput), { code: "ENOENT" });
    await writeFile(input, "{", "utf8");
    await assert.rejects(createReport([input], absentOutput), (error) => error.errors[0].cause instanceof SyntaxError);
    await writeFile(input, "{}", "utf8");
    await assert.rejects(createReport([input], absentOutput), (error) => error.errors[0].cause.message === "file must contain an array");
  } finally {
    const child = relative(root, directory);
    assert.ok(child.startsWith("js-c-test-") && !isAbsolute(child) && !child.includes(".."));
    await rm(directory, { recursive: true });
  }
});
test("report: collect field errors from both input files", async () => {
  const root = import.meta.dirname;
  const directory = await mkdtemp(resolve(root, "js-c-test-"));
  try {
    const inputs = [resolve(directory, "invalid-a.json"), resolve(directory, "invalid-b.json")];
    const output = resolve(directory, "absent.json");
    for (const input of inputs) {
      await writeFile(input, '[{"group":"a","amountCents":-1,"active":true}]', "utf8");
    }
    await assert.rejects(createReport(inputs, output), (error) => {
      assert.ok(error instanceof AggregateError);
      assert.equal(error.errors.length, 2);
      error.errors.forEach((item, index) => {
        assert.equal(item.message, `input ${inputs[index]} failed`);
        assert.ok(item.cause instanceof TypeError);
        assert.equal(item.cause.message, "row 0: amountCents must be nonnegative safe integer");
      });
      return true;
    });
    await assert.rejects(access(output), { code: "ENOENT" });
  } finally {
    const child = relative(root, directory);
    assert.ok(child.startsWith("js-c-test-") && !isAbsolute(child) && !child.includes(".."));
    await rm(directory, { recursive: true });
  }
});
// → 七项测试通过，包括属性读取边界、真实文件读写和没有残留输出的失败路径。
```

## 6 验证实际安装后的消费者

源码相对导入成功，不能证明消费者安装后也能使用：files 可能漏收实现，exports 可能映射错误。npm pack 生成真实 .tgz；再把它安装到全新消费者目录，用包名导入并检查 import.meta.resolve 的位置，才能覆盖分发边界。

下面消费者源码先存放在本章目录，pack-check 会把它复制进临时消费者包再启动新进程。解析位置必须是该消费者自己的 node_modules/notebook-data-summary/index.mjs；还会核对未公开的 validate.mjs 子路径被拒绝。消费者只通过公共接口调用，不靠源仓库相对路径偶然成功。

配套 [consumer.mjs](scripts/26-data-pipeline/consumer.mjs)：

```javascript
import assert from "node:assert/strict";
import { fileURLToPath } from "node:url";
import { resolve } from "node:path";
import { summarizeRows } from "notebook-data-summary";
import { createReport } from "notebook-data-summary/io";

assert.equal(fileURLToPath(import.meta.resolve("notebook-data-summary")), resolve("node_modules/notebook-data-summary/index.mjs"));
assert.equal(typeof createReport, "function");
assert.deepEqual(summarizeRows([{ group: "installed", amountCents: 25, active: true }]), [
  { group: "installed", count: 1, totalCents: 25 },
]);
await assert.rejects(import("notebook-data-summary/validate.mjs"), { code: "ERR_PACKAGE_PATH_NOT_EXPORTED" });
console.log("installed import source verified"); // → installed import source verified
```

## 7 打包、临时安装与清理入口

通过 npm run pack:26 运行时，npm_execpath 指向本次实际 npm CLI。脚本用当前 Node.js 启动它，因此不需要全局工具或拼接 Windows shell 命令。pack --json 提供实际文件清单与包名；本例断言仅包含 package.json 和三个实现文件。

安装使用本地 tarball 和 --offline，确保不会从 registry 下载同名包。--ignore-scripts 禁止该教学包的安装钩子。消费者完成后再移除 tarball、消费者 node_modules 与临时缓存；删除前确认目标在本章目录下且具有本次创建的专用前缀。若任一步失败，保留诊断并仍执行清理。

单独需要研究压缩包内容时，可以读 npm pack 的 JSON 文件清单；教学入口会自行清理，不把构建物当作源码保留。

配套 [pack-check.mjs](scripts/26-data-pipeline/pack-check.mjs)：

```javascript
import { rm } from "node:fs/promises";
import assert from "node:assert/strict";
import { execFileSync } from "node:child_process";
import { mkdtempSync, mkdirSync, writeFileSync, copyFileSync } from "node:fs";
import { resolve, relative, isAbsolute } from "node:path";

const npmCli = process.env.npm_execpath;
assert.ok(npmCli, "run with npm run pack:26");
const root = import.meta.dirname;
const directory = mkdtempSync(resolve(root, "js-c-pack-"));
const packageDirectory = resolve("scripts/26-data-pipeline");
const cache = resolve(directory, "cache");
function npm(args, cwd) {
  return execFileSync(process.execPath, [npmCli, ...args, "--cache", cache], {
    cwd, encoding: "utf8", windowsHide: true,
  });
}
try {
  const [packed] = JSON.parse(npm(["pack", "--json", "--ignore-scripts", "--pack-destination", directory], packageDirectory));
  assert.deepEqual(packed.files.map((file) => file.path).sort(), ["index.mjs", "io.mjs", "package.json", "validate.mjs"]);
  console.log("tarball files", packed.files.length); // → tarball files 4
  const consumer = resolve(directory, "consumer");
  mkdirSync(consumer);
  writeFileSync(resolve(consumer, "package.json"), '{"private":true,"type":"module"}', "utf8");
  npm(["install", "--offline", "--ignore-scripts", "--no-audit", "--no-fund", resolve(directory, packed.filename)], consumer);
  copyFileSync(resolve(packageDirectory, "consumer.mjs"), resolve(consumer, "consumer.mjs"));
  const output = execFileSync(process.execPath, ["consumer.mjs"], { cwd: consumer, encoding: "utf8", windowsHide: true });
  assert.match(output, /installed import source verified/);
  console.log(output.trim()); // → installed import source verified
} finally {
  const child = relative(root, directory);
  assert.ok(child.startsWith("js-c-pack-") && !isAbsolute(child) && !child.includes(".."));
  await rm(directory, { recursive: true });
}
console.log("pack install consumer cleaned"); // → pack install consumer cleaned
```

## 本章小结

- 纯函数定义校验、筛选、分组与排序契约，I/O 接口等待每项副作用完成。
- 自动化测试覆盖正常值、非法数据、精度边界和真实文件失败。
- 包接口还需要实际 tarball 安装消费者验证；源码运行不能替代分发检查。

## 练习

1. 增加一个 inactive 且金额非法的记录；标准：仍然拒绝，因为本例先校验全部记录，再筛选。
2. 将 tools 总额改成与 books 相同；标准：结果按分类名称升序，测试中写出明确期望数组。
3. 在 files 中暂时遗漏 validate.mjs 后运行 npm run pack:26；标准：先在 tarball 文件清单检查处失败，此时尚未安装消费者。恢复清单后命令成功且无临时目录残留。

## 参考与引用来源

- TC39（ECMA-262 第 16 版）：[§23.1 数组与排序](https://tc39.es/ecma262/2025/multipage/indexed-collections.html#sec-array-objects)、[§24.1 Map](https://tc39.es/ecma262/2025/multipage/keyed-collections.html#sec-map-objects)、[§25.5 JSON](https://tc39.es/ecma262/2025/multipage/structured-data.html#sec-json-object)、[§27.2.4.2 allSettled](https://tc39.es/ecma262/2025/multipage/control-abstraction-objects.html#sec-promise.allsettled)：本例使用的语言 API；[OrdinaryGet](https://tc39.es/ecma262/2025/multipage/ordinary-and-exotic-objects-behaviours.html#sec-ordinaryget)：访问器属性的读取会调用 getter；输入格式与排序策略为本例契约。
- Node.js 24.11.0：[fsPromises.rm](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#fspromisesrmpath-options)：临时目录清理；[fs Promises API](https://nodejs.org/download/release/v24.11.0/docs/api/fs.html#promises-api)、[test runner](https://nodejs.org/download/release/v24.11.0/docs/api/test.html)、[包 exports](https://nodejs.org/download/release/v24.11.0/docs/api/packages.html#package-entry-points)、[import.meta.resolve](https://nodejs.org/download/release/v24.11.0/docs/api/esm.html#importmetaresolvespecifier)、[child_process.execFileSync](https://nodejs.org/download/release/v24.11.0/docs/api/child_process.html#child_processexecfilesyncfile-args-options)：真实文件、测试和消费者启动。
- npm：[package.json 的 files、private、exports](https://docs.npmjs.com/cli/v11/configuring-npm/package-json/)、[npm pack](https://docs.npmjs.com/cli/v11/commands/npm-pack/)、[npm install](https://docs.npmjs.com/cli/v11/commands/npm-install/)、[scripts 环境](https://docs.npmjs.com/cli/v11/using-npm/scripts/)：打包、tarball 安装和 npm_execpath。